# Задание 3.1. Система Лоренца: чувствительность к начальным данным
## Зелёный и жёлтый уровни

Исследуется система
$$
\begin{cases}
\dot x=\sigma(y-x),\\
\dot y=x(\rho-z)-y,\\
\dot z=xy-\beta z,
\end{cases}
\qquad
\sigma=10,\quad\beta=\frac83,\quad
\mathbf u(0)=(1,1,1).
$$

Цель работы — проследить расхождение близких решений, вычислить старший показатель Ляпунова при $\rho=10$ и $\rho=28$ и проверить оценку горизонта предсказуемости.

Координаты и время в этой записи **безразмерные**. Все расстояния между состояниями вычисляются в трёхмерном пространстве:
$$d(t)=\|\mathbf v(t)-\mathbf u(t)\|_2
=\sqrt{(v_x-x)^2+(v_y-y)^2+(v_z-z)^2}.$$

В работе различаются три величины:

- $\delta_0$ — начальная ошибка пары траекторий, которая затем свободно эволюционирует;
- $\varepsilon$ — малое расстояние, восстанавливаемое при вычислении показателя Ляпунова;
- $\Delta$ — допустимая ошибка прогноза, момент первого достижения которой требуется найти.

Порядок разделов: численное решение → опыт без перенормировки → показатель Ляпунова → проверка параметров → горизонт предсказуемости → анимация и выводы.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Markdown, display

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 45,
    "animation.frame_format": "jpeg",
})

sigma, beta = 10.0, 8.0/3.0
u_initial = np.array([1.0, 1.0, 1.0])
ex = np.array([1.0, 0.0, 0.0])
C = {"base": "#27677c", "perturbed": "#c26a36",
     "regular": "#797044", "chaotic": "#574c9a", "numeric": "#909090"}


def table(headers, rows):
    def row(values):
        return "| " + " | ".join(str(v) for v in values) + " |"
    text = "\n".join([row(headers), row(["---"]*len(headers))]
                     + [row(values) for values in rows])
    display(Markdown(text))

## 1. Численное решение и движение при $\rho=28$

Используем классический метод Рунге–Кутты четвёртого порядка с постоянным шагом:
$$
\begin{aligned}
k_1&=f(\mathbf u_n),&
k_2&=f(\mathbf u_n+hk_1/2),\\
k_3&=f(\mathbf u_n+hk_2/2),&
k_4&=f(\mathbf u_n+hk_3),\\
\mathbf u_{n+1}&=\mathbf u_n+\frac h6(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$
При уменьшении $h$ локальная ошибка каждого шага уменьшается. Однако на длинном хаотическом движении ранее накопленные ошибки усиливаются, поэтому совпадение отдельных траекторий на произвольно больших временах не ожидается.

Правая часть ниже работает и с одним вектором $(x,y,z)$, и с массивом таких векторов.

In [ ]:
def lorenz(u, rho):
    x, y, z = u[..., 0], u[..., 1], u[..., 2]
    return np.stack((sigma*(y-x), x*(rho-z)-y, x*y-beta*z), axis=-1)


def rk4_step(u, h, rho):
    k1 = lorenz(u, rho)
    k2 = lorenz(u + 0.5*h*k1, rho)
    k3 = lorenz(u + 0.5*h*k2, rho)
    k4 = lorenz(u + h*k3, rho)
    return u + h*(k1 + 2*k2 + 2*k3 + k4)/6


def number_of_steps(duration, h):
    n = int(round(duration/h))
    if h <= 0 or n < 1 or not np.isclose(n*h, duration, rtol=0, atol=1e-9):
        raise ValueError("Продолжительность должна быть положительной и кратной шагу.")
    return n


def solve(u0, rho, duration, h):
    n = number_of_steps(duration, h)
    u0 = np.asarray(u0, dtype=float)
    values = np.empty((n+1,) + u0.shape)
    values[0] = u0
    for k in range(n):
        values[k+1] = rk4_step(values[k], h, rho)
    return h*np.arange(n+1), values


h_reference = 0.0005
t_full, orbit = solve(u_initial, 28.0, 60.0, h_reference)
t_transient = 20.0
start_index = int(round(t_transient/h_reference))
phase = orbit[start_index::10]  # прореживание только для рисунка
start28 = orbit[start_index].copy()
_, regular_transient = solve(u_initial, 10.0, t_transient, h_reference)
start10 = regular_transient[-1].copy()
starts = np.stack((start10, start28))
rhos = np.array([10.0, 28.0])

print("Общая точка начала парных опытов при ρ = 28:")
print(np.array2string(start28, precision=10))
print("Размахи координат после t = 20:",
      np.array2string(np.ptp(phase, axis=0), precision=2))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10.5, 6), sharex=True,
                         constrained_layout=True)
for j, (ax, name, color) in enumerate(zip(
        axes, ["x", "y", "z"], [C["base"], C["perturbed"], C["chaotic"]])):
    ax.axvspan(0, t_transient, color="#e5e5e5")
    ax.plot(t_full[::10], orbit[::10, j], color=color, lw=0.8)
    ax.set_ylabel(name)
    ax.grid(alpha=0.2)
axes[-1].set(xlabel="t", xlim=(0, 60))
fig.suptitle("Временные ряды при ρ = 28; серым отмечен переходный участок")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.9), constrained_layout=True)
for ax, (i, j) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
    ax.plot(phase[:, i], phase[:, j], color=C["base"], lw=0.5)
    ax.set(xlabel=["x", "y", "z"][i], ylabel=["x", "y", "z"][j], aspect="equal")
    ax.grid(alpha=0.2)
fig.suptitle("Фазовый портрет: три проекции при 20 ≤ t ≤ 60")
plt.show()

Движение остаётся в ограниченной области с двумя характерными лепестками.

## 2. Свободное расхождение близких траекторий — зелёный уровень

Из одной сохранённой точки $\mathbf u_*=\mathbf u(20)$ запускаются два решения:
$$\mathbf u(0)=\mathbf u_*,
\qquad\mathbf v(0)=\mathbf u_*+10^{-6}\mathbf e_x.$$
 Время $s$ отсчитывается от внесения возмущения.

Сравним $d_h(s)$ для трёх шагов $h=0.004,\ 0.002,\ 0.001$. Во всех случаях передаётся один и тот же массив $\mathbf u_*$. На общей сетке дополнительно измерим
$$e_{h,h/2}(s)=\|\mathbf u_h(s)-\mathbf u_{h/2}(s)\|.$$
Это различие двух численных решений из **одинакового** состояния. Точное решение неизвестно, поэтому $e_{h,h/2}$ служит диагностикой влияния шага.

In [ ]:
green_delta = 1e-6
green_steps = [0.004, 0.002, 0.001]
green_runs = {}
pair0 = np.stack((start28, start28 + green_delta*ex))
for h in green_steps:
    s, pair = solve(pair0, 28.0, 32.0, h)
    green_runs[h] = {"s": s, "pair": pair,
                     "distance": np.linalg.norm(pair[:, 1] - pair[:, 0], axis=1)}

s_common = green_runs[0.004]["s"]
aligned = {}
for h in green_steps:
    stride = int(round(0.004/h))
    aligned[h] = green_runs[h]["pair"][::stride]
d_common = {h: np.linalg.norm(q[:, 1]-q[:, 0], axis=1)
            for h, q in aligned.items()}
e_coarse = np.linalg.norm(aligned[0.004][:, 0]-aligned[0.002][:, 0], axis=1)
e_fine = np.linalg.norm(aligned[0.002][:, 0]-aligned[0.001][:, 0], axis=1)
fit_mask = (s_common >= 2) & (s_common <= 12)
rate, offset = np.polyfit(s_common[fit_mask],
                         np.log(d_common[0.001][fit_mask]), 1)

table(["Проверка на 2 ≤ s ≤ 12", "Результат"], [
    ["Средний наклон ln d", f"{rate:.4f}"],
    ["max e(0.002, 0.001) / d(0.001)",
     f"{np.max(e_fine[fit_mask]/d_common[0.001][fit_mask]):.3e}"],
    ["max abs[d(0.002) − d(0.001)] / d(0.001)",
     f"{np.max(np.abs(d_common[0.002][fit_mask]-d_common[0.001][fit_mask]) / d_common[0.001][fit_mask]):.3e}"],
])

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.5), constrained_layout=True)
for h, color in zip(green_steps, [C["numeric"], C["perturbed"], C["base"]]):
    axes[0].semilogy(s_common, d_common[h], color=color, lw=1.2, label=f"h = {h:g}")
axes[0].axvspan(2, 12, color="#e6dbbf", alpha=0.4)
axes[0].semilogy(s_common[fit_mask], np.exp(offset+rate*s_common[fit_mask]),
                 "--", color="black", lw=1.3, label=f"Тренд exp({rate:.2f} s)")
axes[0].set(xlabel="s", ylabel="Расстояние d", title="Возмущение только по x; без перенормировки")
axes[0].legend(fontsize=8, loc="lower right")
axes[1].semilogy(s_common, d_common[0.001], color=C["base"], lw=1.7,
                 label="Пара с начальным возмущением")
for error, label, color in [(e_coarse, "Одинаковый старт: h = 0.004 и 0.002", C["numeric"]),
                             (e_fine, "Одинаковый старт: h = 0.002 и 0.001", C["perturbed"])]:
    axes[1].semilogy(s_common, np.where(error > 0, error, np.nan),
                     color=color, lw=1.1, label=label)
axes[1].set(xlabel="s", ylabel="Расстояние", title="Сопоставление с влиянием шага")
axes[1].legend(fontsize=8, loc="lower right")
for ax in axes:
    ax.set_xlim(0, 32)
    ax.grid(which="both", alpha=0.2)
plt.show()

На участке $2\le s\le12$ средний рост близок к экспоненциальному; поверх него видны растяжения и сжатия. Его воспроизводимость при уменьшении шага проверяется таблицей и графиком. Наклон прямой на одном коротком участке зависит от начальной точки и выбранного окна, поэтому его ещё нельзя считать асимптотическим показателем Ляпунова.

Когда расстояние достигает масштаба аттрактора, линейное приближение по возмущению перестаёт работать. Обе траектории продолжают двигаться в ограниченной области, поэтому неограниченный экспоненциальный рост невозможен. Расстояние после этого колеблется, а не становится постоянным.

## 3. Старший показатель Ляпунова — жёлтый уровень

### 3.1. Зачем нужна перенормировка

Для бесконечно малого возмущения старший показатель описывает средний темп роста его нормы:
$$\lambda_{\max}=\lim_{T\to\infty}\frac1T
\ln\frac{\|\boldsymbol\eta(T)\|}{\|\boldsymbol\eta(0)\|}.$$
Здесь $\boldsymbol\eta$ — возмущение в линейном приближении;  У конечной свободной пары рост насыщается, и отношение $\ln[d(T)/d(0)]/T$ при большом $T$ уже не измеряет искомый показатель.

Поддержим возмущение малым. После каждого интервала $\tau$:

1. Интегрируем опорную и вспомогательную траектории.
2. Вычисляем вектор $\mathbf w_k=\mathbf v_k-\mathbf u_k$ и его длину $d_k$.
3. Накапливаем логарифм коэффициента растяжения.
4. Меняем только вспомогательное состояние:
   $$\mathbf v_k\leftarrow\mathbf u_k+\varepsilon\frac{\mathbf w_k}{d_k}.$$

Сохраняется текущее направление возмущения; оно не возвращается каждый раз к оси $x$. Опорная траектория при перенормировке не изменяется.

В идеальной арифметике стартовая длина каждого интервала равна $\varepsilon$. В коде используем её фактическое значение $b_k$ после округления:
$$\widehat\lambda(T)=\frac{1}{T}\sum_{k=1}^m
\ln\frac{d_k}{b_k},\qquad T=m\tau.$$
Так произведение коэффициентов роста заменяется суммой логарифмов и не переполняет числовой диапазон.

Основные параметры: $\varepsilon=10^{-7}$, $\tau=0.1$, $h=0.005$, $T=800$. После перехода до $t=20$ ещё 20 единиц времени выполняются с перенормировкой для ориентации возмущения; эти интервалы не включаются в среднее. На графиках $T$ отсчитывается от начала накопления суммы.

Сразу рассчитываются два значения $\rho$. Внутри массива ось 0 обозначает режим ($10$ или $28$), ось 1 — траекторию (опорная или вспомогательная), ось 2 — координату.

In [ ]:
def estimate_lyapunov(starts, rhos, duration=800.0, epsilon=1e-7,
                      tau=0.1, h=0.005, alignment=20.0):
    substeps = number_of_steps(tau, h)
    n_align = number_of_steps(alignment, tau)
    n_average = number_of_steps(duration, tau)
    pair = np.stack((starts.copy(), starts + epsilon*ex), axis=1)
    rho_for_pairs = np.asarray(rhos)[:, None]
    log_growth = np.empty((n_average, len(rhos)))

    for block in range(n_align + n_average):
        start_distance = np.linalg.norm(pair[:, 1]-pair[:, 0], axis=1)
        for _ in range(substeps):
            pair = rk4_step(pair, h, rho_for_pairs)

        displacement = pair[:, 1]-pair[:, 0]
        distance = np.linalg.norm(displacement, axis=1)
        if np.any(distance <= 0) or not np.isfinite(distance).all():
            raise FloatingPointError("Расстояние между траекториями не разрешается.")
        if block >= n_align:
            log_growth[block-n_align] = np.log(distance/start_distance)
        pair[:, 1] = pair[:, 0] + epsilon*displacement/distance[:, None]

    times = tau*np.arange(1, n_average+1)
    return {
        "time": times,
        "logs": log_growth,
        "estimate": np.cumsum(log_growth, axis=0)/times[:, None],
        "tau": tau, "h": h, "epsilon": epsilon,
    }


long_run = estimate_lyapunov(starts, rhos)


def estimate_at(run, time):
    index = int(round(time/run["tau"])) - 1
    return run["estimate"][index]


averaging_times = [50, 100, 200, 400, 800]
table(["Время усреднения T", "λ̂ при ρ = 10", "λ̂ при ρ = 28"], [
    [T, f"{estimate_at(long_run, T)[0]:.5f}",
     f"{estimate_at(long_run, T)[1]:.5f}"] for T in averaging_times
])

fig, axes = plt.subplots(1, 2, figsize=(11.7, 4.5), constrained_layout=True)
for j, color in enumerate([C["regular"], C["chaotic"]]):
    mask = long_run["time"] >= 10
    axes[0].plot(long_run["time"][mask], long_run["estimate"][mask, j],
                 color=color, lw=1.2, label=f"ρ = {rhos[j]:g}")
axes[0].axhline(0, color="gray", lw=0.8)
axes[0].set(xlabel="Время усреднения T", ylabel="λ̂(T)", title="Два режима")
axes[0].legend()
mask = long_run["time"] >= 100
axes[1].plot(long_run["time"][mask], long_run["estimate"][mask, 1],
             color=C["chaotic"], lw=1.1)
axes[1].set(xlabel="Время усреднения T", ylabel="λ̂(T)",
            title="Увеличенный поздний участок при ρ = 28")
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()



### 3.2. Устойчивость оценки

Проверим три требования из условия:

| Что меняется | Как проверяется |
|---|---|
| Длина возмущения | $\varepsilon=10^{-5},10^{-7},10^{-8}$ |
| Интервал перенормировки | $\tau=0.05,0.1,0.2$ |
| Точность интегрирования | $h=0.01,0.005,0.0025$ |

В следующем сравнении время усреднения одинаково: $T=400$. При каждой проверке меняется один параметр относительно основного варианта. Стартовые состояния и предварительная ориентация одинаковы.

Для хаотического режима оценки на конечном интервале могут различаться даже при качественном интегрировании: малое изменение вычисления меняет посещаемые участки аттрактора. Чтобы оценить масштаб этой изменчивости, отдельно рассмотрим четыре неперекрывающихся блока длиной 200 одной основной реализации. Их разброс не является доверительным интервалом.

In [ ]:
variants = [
    ("Основной",                 1e-7, 0.10, 0.0050),
    ("Большее возмущение",       1e-5, 0.10, 0.0050),
    ("Меньшее возмущение",       1e-8, 0.10, 0.0050),
    ("Чаще перенормировка",      1e-7, 0.05, 0.0050),
    ("Реже перенормировка",      1e-7, 0.20, 0.0050),
    ("Увеличенный шаг RK4",      1e-7, 0.10, 0.0100),
    ("Уменьшенный шаг RK4",      1e-7, 0.10, 0.0025),
]
robustness = []
for name, eps, tau, h in variants:
    if name == "Основной":
        values = estimate_at(long_run, 400)
    else:
        result = estimate_lyapunov(starts, rhos, duration=400.0,
                                   epsilon=eps, tau=tau, h=h)
        values = result["estimate"][-1]
    robustness.append(values)
robustness = np.array(robustness)

table(["Вариант", "ε", "τ", "h", "λ̂, ρ = 10", "λ̂, ρ = 28"], [
    [name, f"{eps:.0e}", f"{tau:g}", f"{h:g}", f"{value[0]:.5f}", f"{value[1]:.5f}"]
    for (name, eps, tau, h), value in zip(variants, robustness)
])

blocks = long_run["logs"].reshape(4, -1, 2).sum(axis=1)/200.0
table(["Интервал усреднения", "Средняя скорость, ρ = 10", "Средняя скорость, ρ = 28"], [
    [f"{k*200}–{(k+1)*200}", f"{value[0]:.5f}", f"{value[1]:.5f}"]
    for k, value in enumerate(blocks)
])
display(Markdown(
    f"При изменении параметров оценка для $\\rho=28$ находится в пределах "
    f"**{robustness[:, 1].min():.3f}–{robustness[:, 1].max():.3f}**. "
    f"Блоки одной длинной реализации дают "
    f"**{blocks[:, 1].min():.3f}–{blocks[:, 1].max():.3f}**. "
    "Сопоставление этих диапазонов помогает не приписывать весь разброс "
    "одной лишь ошибке интегрирования. Последние десятичные знаки "
    "конечного среднего не следует считать установленной точностью показателя."
))

Слишком большое $\varepsilon$ выводило бы пару из линейного режима, а слишком малое делало бы вычитание двух близких векторов чувствительным к округлению. Короткий $\tau$ увеличивает число таких вычитаний; слишком длинный $\tau$ может допустить чрезмерный рост расстояния. В выбранном диапазоне параметров проверяются знак и характерная величина показателя, а не гарантируется произвольная точность.

## 4. Горизонт предсказуемости

### 4.1. Теоретическая оценка

Если в среднем $d(s)\approx\delta_0 e^{\lambda_{\max}s}$, то из условия $d(T_{\rm pred})=\Delta$:
$$T_{\rm pred}\approx\frac{\ln(\Delta/\delta_0)}{\lambda_{\max}}.$$

Выберем **$\Delta=0.5$**. Размахи координат в разделе 1 составляют десятки единиц, поэтому этот порог мал относительно размера аттрактора. Формула использует долговременную среднюю скорость роста; первая потеря точности конкретной пары зависит также от направления возмущения и участка движения.

Возьмём девять стартовых точек исходной траектории при $t=20,25,\ldots,60$. Для каждой зададим пять ошибок $\delta_0=10^{-3},10^{-4},10^{-5},10^{-6},10^{-7}$ только по $x$. Всего получается 45 пар. После запуска перенормировка **не применяется**.

Ищется первое достижение порога. Если между узлами $s_n$ и $s_n+h$ выполнено $d_n<\Delta\le d_{n+1}$, используем линейное уточнение:
$$T_{\rm hit}\approx s_n+
h\frac{\Delta-d_n}{d_{n+1}-d_n}.$$
Это приближение, точность которого ниже проверяется уменьшением шага.

Все пары можно интегрировать одним массивом: дополнительные оси обозначают стартовую точку и величину возмущения. Начальный массив у каждого уточнённого расчёта в точности тот же.

In [ ]:
def first_passages(points, deltas, threshold=0.5, h=0.001, max_time=45.0):
    base = np.broadcast_to(points[:, None, :],
                            (len(points), len(deltas), 3)).copy()
    shifted = base + deltas[None, :, None]*ex
    pair = np.stack((base, shifted), axis=-2)
    previous = np.linalg.norm(pair[..., 1, :]-pair[..., 0, :], axis=-1)
    hits = np.full(previous.shape, np.nan)

    for k in range(number_of_steps(max_time, h)):
        pair = rk4_step(pair, h, 28.0)
        distance = np.linalg.norm(pair[..., 1, :]-pair[..., 0, :], axis=-1)
        crossed = np.isnan(hits) & (previous < threshold) & (distance >= threshold)
        fraction = (threshold-previous[crossed])/(distance[crossed]-previous[crossed])
        hits[crossed] = (k + fraction)*h
        if np.isfinite(hits).all():
            return hits
        previous = distance

    raise RuntimeError("Некоторые пары не достигли порога: увеличьте max_time.")


start_times = np.arange(20.0, 60.1, 5.0)
horizon_points = orbit[np.rint(start_times/h_reference).astype(int)].copy()
deltas = 10.0**(-np.arange(3, 8))
Delta = 0.5
hit_times = first_passages(horizon_points, deltas, Delta, h=0.001)
hit_times_refined = first_passages(horizon_points, deltas, Delta, h=0.0005)
hit_change = np.abs(hit_times_refined-hit_times)

# Для дальнейших графиков используем уточнённый расчёт.
measured = hit_times_refined
mean_hits = measured.mean(axis=0)
spread_hits = measured.std(axis=0, ddof=1)
theory_hits = np.log(Delta/deltas)/lambda28

table(["δ₀", "Минимум T", "Среднее T", "Максимум T", "Станд. отклонение",
       "Теория", "max abs[T(h) − T(h/2)]"], [
    [f"{d:.0e}", f"{column.min():.3f}", f"{column.mean():.3f}",
     f"{column.max():.3f}", f"{column.std(ddof=1):.3f}",
     f"{theory:.3f}", f"{change.max():.2e}"]
    for d, column, theory, change in zip(deltas, measured.T, theory_hits, hit_change.T)
])

fig, ax = plt.subplots(figsize=(9.5, 5.1), constrained_layout=True)
for k, values in enumerate(measured):
    ax.semilogx(deltas, values, color="#b8b8b8", marker=".", ms=4, lw=0.9,
                 label="Отдельные стартовые точки" if k == 0 else None)
ax.errorbar(deltas, mean_hits, yerr=spread_hits, fmt="o-",
             color=C["base"], lw=2, capsize=4, label="Среднее ± станд. отклонение")
ax.semilogx(deltas, theory_hits, "--", color=C["perturbed"], lw=2,
             label="ln(Δ/δ₀) / λ̂(800)")
ax.invert_xaxis()
ax.set(xlabel="Начальная ошибка δ₀", ylabel="Первое время достижения Δ",
       title="Горизонт предсказуемости: Δ = 0.5, девять стартовых точек")
ax.grid(which="both", alpha=0.2)
ax.legend(fontsize=9)
plt.show()

Стандартное отклонение на рисунке описывает разброс **между выбранными начальными точками**. Оно не является оценкой ошибки интегрирования или гарантированной границей времени прогноза.

### 4.2. Что даёт десятикратное уточнение начального состояния

При замене $\delta_0$ на $\delta_0/10$ теоретическое увеличение горизонта:
$$
\Delta T=
\frac{\ln(\Delta/(\delta_0/10))-\ln(\Delta/\delta_0)}
{\lambda_{\max}}
=\frac{\ln10}{\lambda_{\max}}.
$$
Таким образом, выигрыш **прибавляется ко времени** и не зависит от исходной $\delta_0$ в приближении постоянной средней скорости роста. Для отдельной пары локальные колебания растяжения дают отклонения от этой оценки.

In [ ]:
gain_theory = np.log(10)/lambda28
measured_gains = np.diff(measured, axis=1)

table(["Уточнение начальных данных", "Средняя прибавка T", "Станд. отклонение"], [
    [f"{deltas[k]:.0e} → {deltas[k+1]:.0e}",
     f"{measured_gains[:, k].mean():.3f}",
     f"{measured_gains[:, k].std(ddof=1):.3f}"]
    for k in range(len(deltas)-1)
])
display(Markdown(
    f"По показателю $\\widehat\\lambda={lambda28:.4f}$ "
    f"десятикратное уточнение добавляет примерно **{gain_theory:.2f} единицы времени**. "
    f"Средняя прибавка по всем проверенным переходам равна "
    f"**{measured_gains.mean():.2f}**. "
    f"Максимальное изменение времени первого пересечения при делении шага пополам: "
    f"**{hit_change.max():.3g}**."
))

fig, ax = plt.subplots(figsize=(8.5, 4.3), constrained_layout=True)
positions = np.arange(len(deltas)-1)
ax.errorbar(positions, measured_gains.mean(axis=0),
             yerr=measured_gains.std(axis=0, ddof=1), fmt="o",
             color=C["base"], capsize=5, label="Средняя прибавка ± станд. отклонение")
ax.axhline(gain_theory, color=C["perturbed"], ls="--", label="ln(10) / λ̂(800)")
ax.set(xticks=positions,
       xticklabels=[f"{deltas[k]:.0e} → {deltas[k+1]:.0e}" for k in positions],
       xlabel="Уменьшение начальной ошибки в 10 раз",
       ylabel="Прибавка времени", title="Выигрыш от более точных начальных данных")
ax.grid(alpha=0.2)
ax.legend(fontsize=9)
plt.show()

### 4.3. Как интерпретировать расхождения с формулой

Формула с $\lambda_{\max}$ описывает среднюю тенденцию. В начале возмущение ориентировано по $x$ и ещё не совпадает с наиболее растягивающим направлением; затем оно проходит участки с разной интенсивностью роста. Поэтому первое пересечение $\Delta$ не обязано точно совпадать с теоретическим временем, а отдельная пара может потерять заданную точность раньше или позже среднего.

Сравнение $h=0.001$ и $h=0.0005$ относится ко всем 45 парам. Оно проверяет одновременно интегрирование и линейное уточнение момента пересечения. Числа изменения времени следует сопоставлять с разбросом между стартовыми точками из таблицы, а не с длиной шага напрямую.

При $\rho=10$ показатель отрицателен: рядом с притягивающим равновесием малые ошибки затухают. Формулу положительного времени потери точности вследствие экспоненциального роста к этому режиму не применяем.

## 5. Анимация двух траекторий и их расстояния

Для анимации используется свободная пара из раздела 2 с $\delta_0=10^{-6}$. Слева показана проекция $(x,z)$ и след за последние 2 единицы времени, справа — расстояние **в полном трёхмерном пространстве**. Движущиеся отметки соответствуют одному моменту времени.

На правой панели нанесён выбранный порог $\Delta=0.5$. Перенормировки в анимации нет. Пока расстояние мало, два следа визуально совпадают; логарифмический график позволяет увидеть рост задолго до заметного расхождения в проекции.

In [ ]:
animation_run = green_runs[0.001]
animation_t = animation_run["s"]
animation_pair = animation_run["pair"]
animation_d = animation_run["distance"]
frames = np.linspace(0, len(animation_t)-1, 151, dtype=int)
trail = int(round(2.0/0.001))

fig, (ax_orbit, ax_distance) = plt.subplots(1, 2, figsize=(10.3, 4.3))
fig.subplots_adjust(left=0.065, right=0.98, bottom=0.15, top=0.86, wspace=0.28)
ax_orbit.plot(phase[:, 0], phase[:, 2], color="#d7d7d7", lw=0.4)
base_line, = ax_orbit.plot([], [], color=C["base"], lw=2.3, label="Опорная")
pert_line, = ax_orbit.plot([], [], color=C["perturbed"], lw=1.5,
                           ls="--", label="Возмущённая")
base_dot, = ax_orbit.plot([], [], "o", color=C["base"], ms=6)
pert_dot, = ax_orbit.plot([], [], "s", mfc="none", mec=C["perturbed"], mew=1.5, ms=7)
ax_orbit.set(xlabel="x", ylabel="z", xlim=(-22, 22), ylim=(0, 50),
             aspect="equal", title="Два состояния в проекции (x, z)")
ax_orbit.legend(fontsize=8, loc="upper right")
distance_line, = ax_distance.semilogy([], [], color=C["chaotic"], lw=1.5)
distance_dot, = ax_distance.semilogy([], [], "o", color=C["chaotic"], ms=5)
cursor = ax_distance.axvline(0, color="gray", ls=":", lw=1)
ax_distance.axhline(Delta, color=C["perturbed"], ls="--", label="Δ = 0.5")
ax_distance.set(xlabel="s — время после возмущения", ylabel="d(s)",
                xlim=(0, 32), ylim=(1e-8, 100), title="Расстояние без перенормировки")
ax_distance.legend(fontsize=8, loc="lower right")
for ax in (ax_orbit, ax_distance):
    ax.grid(alpha=0.2)
heading = fig.suptitle("")


def update_frame(frame_number):
    k = frames[frame_number]
    first = max(0, k-trail)
    base_line.set_data(animation_pair[first:k+1, 0, 0], animation_pair[first:k+1, 0, 2])
    pert_line.set_data(animation_pair[first:k+1, 1, 0], animation_pair[first:k+1, 1, 2])
    base_dot.set_data([animation_pair[k, 0, 0]], [animation_pair[k, 0, 2]])
    pert_dot.set_data([animation_pair[k, 1, 0]], [animation_pair[k, 1, 2]])
    distance_line.set_data(animation_t[:k+1], animation_d[:k+1])
    distance_dot.set_data([animation_t[k]], [animation_d[k]])
    cursor.set_xdata([animation_t[k], animation_t[k]])
    heading.set_text(f"ρ = 28;  s = {animation_t[k]:.2f};  d = {animation_d[k]:.3g}")
    return base_line, pert_line, base_dot, pert_dot, distance_line, distance_dot, cursor, heading


movie = FuncAnimation(fig, update_frame, frames=len(frames), interval=75, blit=False)
with plt.rc_context({"savefig.dpi": 80}):
    animation_html = movie.to_jshtml(fps=14, default_mode="once")
plt.close(fig)
display(HTML(animation_html))

## 6. Выводы и пояснения к результатам

**Рост свободного возмущения.** При $\rho=28$ участок $2\le s\le12$ демонстрирует приблизительно экспоненциальный средний рост расстояния. Проверка нескольких шагов позволяет отделить его от влияния точности интегрирования. Неограниченный рост невозможен: обе траектории остаются в ограниченной области, а при больших расстояниях приближение малого возмущения перестаёт работать.

**Старший показатель.** Перенормировка сохраняет малость возмущения, а сумма логарифмов сохраняет информацию о его растяжении до каждого уменьшения. При $\rho=10$ получается отрицательная величина около $-0.595$, согласующаяся с притяжением к равновесию. При $\rho=28$ получается положительная величина порядка $0.9$, соответствующая чувствительности к начальным условиям.

**Устойчивость оценки.** Проверены увеличение времени усреднения, длина возмущения, интервал перенормировки и шаг RK4. Длинное среднее сходится с колебаниями. Сравнение настроек и блоков одной реализации ограничивает количество значащих цифр, с которым разумно сообщать результат.

**Время прогноза.** По 45 свободным парам проверена зависимость первого достижения $\Delta=0.5$ от начальной ошибки. Теоретическая формула описывает общий логарифмический рост горизонта; конкретная пара может существенно отклоняться из-за неодинакового локального растяжения.

**Ответ на вопрос задания.** Десятикратное уточнение начального состояния добавляет $\ln10/\lambda_{\max}$, то есть примерно $2.5$ единицы времени в хаотическом режиме. Точное численное значение по полученной оценке и прямое сравнение приведены в разделе 4.2.
